# BusNet speed benchmark
Times a full **training step** (forward + CE + backward + optimiser) of the field BusNet under a ladder of optimisations, checks each against eager fp32 for correctness, and prints the config to adopt. Run on the 4090 box; `SCALED=True` benches the M=8/d=8 config, `False` the base d=6 cell. Numbers printed in this checked-in copy come from the CPU sandbox at toy sizes; regenerate locally.

In [1]:
import time, types, torch, torch.nn.functional as F
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.models.busnet import BusNet, BusNetConfig
from src.tasks.sort_of_clevr.data import constants as CN

SCALED = True
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
BS = 256 if DEV == 'cuda' else 8
ITERS, WARM = (30, 10) if DEV == 'cuda' else (2, 1)
torch.manual_seed(0)
kw = dict(name='b', encoder={'name': 'field'}, per_module_gru=False, phase_repr='vector', drive='stimulus')
kw |= dict(n_modules=8, osc_dim=8, msg_dim=8, module_dim=128, T=12, field_T=12, slot_iters=4) if SCALED else dict(osc_dim=6)
x = torch.randn(BS, 3, 75, 75, device=DEV)
q = torch.zeros(BS, 18, device=DEV); q[:, 0] = 1; q[:, 6] = 1
y = torch.randint(0, 10, (BS,), device=DEV)
print(f'device={DEV} bs={BS} scaled={SCALED}')

device=cuda bs=256 scaled=True


In [2]:
def make():
    torch.manual_seed(0)
    return BusNet(BusNetConfig(**kw), 75, 10, list(CN.COLOURS.values())).to(DEV)

def bench(step, model):
    for _ in range(WARM): step(model)
    if DEV == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(ITERS): step(model)
    if DEV == 'cuda': torch.cuda.synchronize()
    return (time.time() - t0) / ITERS * 1000  # ms/step

def train_step(autocast):
    def step(model):
        model.opt.zero_grad(set_to_none=True)
        with torch.autocast(DEV, dtype=torch.bfloat16, enabled=autocast):
            loss = F.cross_entropy(model(x, q)['logits'], y)
        loss.backward(); model.opt.step()
    return step

@torch.no_grad()
def logits_of(model):
    model.eval(); out = model(x, q)['logits'].float().clone(); model.train(); return out

_ref_model = make()
REF = logits_of(_ref_model)
def new(label, wrapper=lambda m: m, autocast=True):
    m = make(); m.opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
    wm = wrapper(m); wm.opt = m.opt if wm is not m else m.opt
    d = (logits_of(m) - REF).abs().max().item()
    results[label] = (bench(train_step(autocast), wm), d)
results = {}

## Ladder of variants
Each row: fresh model, identical init, one change added. `logit drift` is max |Δ| vs eager fp32 on a fixed batch before training — bf16/compile reorder float ops, so ~1e-2 under autocast and ~1e-3 under compile are expected and harmless.

In [3]:
new('eager fp32', autocast=False)
new('bf16 autocast')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
new('bf16 + tf32 + cudnn.benchmark')

In [4]:
for mode, label in [(None, '+ compile (default)'), ('max-autotune', '+ compile (max-autotune)')]:
    try:
        new(label, wrapper=lambda m, md=mode: torch.compile(m, mode=md, dynamic=False))
    except Exception as e:
        results[label] = (float('nan'), float('nan')); print(label, 'failed:', type(e).__name__, str(e)[:120])

W0829 14:12:47.986000 2399833 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] Graph break from `Tensor.item()`, consider setting:
W0829 14:12:47.986000 2399833 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W0829 14:12:47.986000 2399833 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] or:
W0829 14:12:47.986000 2399833 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W0829 14:12:47.986000 2399833 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] to include these operations in the captured graph.
W0829 14:12:47.986000 2399833 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] 
W0829 14:12:47.986000 2399833 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] Graph break: from user code at:
W0829 14:12:47.986000 2399833 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0]   File "/home/nik/workspace/ImperialWork

+ compile (max-autotune) failed: AcceleratorError CUDA error: operation failed due to a previous error during capture
Search for `cudaErrorStreamCaptureInvalidated' in ht


In [5]:
# rotate() as a block-diagonal 1x1 conv instead of einsum (candidate kernel swap in the field)
def rotate_conv(self, z):
    A = (self.omega_raw - self.omega_raw.transpose(-1, -2)) * self.omega_scale   # (K, d, d)
    Wf = torch.block_diag(*A.unbind(0)).view(self.C, self.C, 1, 1).to(z.dtype)
    return F.conv2d(z, Wf)
try:
    m = make()
    osc = m.field                                    # BusNet's OscillatorField
    a = torch.randn(2, osc.C, 19, 19, device=DEV)
    ok = torch.allclose(osc.rotate(a), rotate_conv(osc, a), atol=1e-5)
    print('grouped-conv rotate equivalent:', ok)
    if ok:
        m.opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
        osc.rotate = types.MethodType(rotate_conv, osc)
        d = (logits_of(m) - REF).abs().max().item()
        results['bf16 + conv rotate'] = (bench(train_step(True), m), d)
except Exception as e:
    print('rotate swap skipped:', type(e).__name__, str(e)[:140])

grouped-conv rotate equivalent: False


In [6]:
# throughput vs batch size at plain bf16 (samples/sec)
for bs2 in ([128, 256, 512] if DEV == 'cuda' else [8]):
    x2 = torch.randn(bs2, 3, 75, 75, device=DEV)
    q2 = torch.zeros(bs2, 18, device=DEV); q2[:, 0] = 1; q2[:, 6] = 1
    y2 = torch.randint(0, 10, (bs2,), device=DEV)
    m = make(); m.opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
    def step(model, x2=x2, q2=q2, y2=y2):
        model.opt.zero_grad(set_to_none=True)
        with torch.autocast(DEV, dtype=torch.bfloat16, enabled=True):
            loss = F.cross_entropy(model(x2, q2)['logits'], y2)
        loss.backward(); model.opt.step()
    ms = bench(step, m)
    print(f'bs={bs2:4d}  {ms:7.1f} ms/step  {bs2/ms*1000:8.0f} samples/s')

bs= 128     64.8 ms/step      1976 samples/s
bs= 256     81.2 ms/step      3151 samples/s
bs= 512    136.7 ms/step      3746 samples/s


In [7]:
print(f"{'variant':32s} {'ms/step':>9s} {'x vs fp32':>10s} {'logit drift':>12s}")
b0 = results['eager fp32'][0]
for k, (ms, d) in results.items():
    print(f'{k:32s} {ms:9.1f} {b0/ms:9.2f}x {d:12.2e}')
print('''
Adopt (uniformly across all body cells, per the fairness rule):
  train.compile_model=true          # if the compile rows win and drift stays small
  + in main.py, before model build:
      torch.backends.cuda.matmul.allow_tf32 = True
      torch.backends.cudnn.allow_tf32 = True
      torch.backends.cudnn.benchmark = True
  logging.eval_log_interval=2000    # for 200k runs (logging cadence, not the model)
If "conv rotate" wins by >10%, say the word and osc_core gets the swap behind an
exact-equivalence test, so it is the same model to the bit.''')

variant                            ms/step  x vs fp32  logit drift
eager fp32                           156.8      1.00x     0.00e+00
bf16 autocast                         76.8      2.04x     0.00e+00
bf16 + tf32 + cudnn.benchmark         76.7      2.04x     4.46e-06
+ compile (default)                   33.0      4.75x     4.46e-06
+ compile (max-autotune)               nan       nanx          nan

Adopt (uniformly across all body cells, per the fairness rule):
  train.compile_model=true          # if the compile rows win and drift stays small
  + in main.py, before model build:
      torch.backends.cuda.matmul.allow_tf32 = True
      torch.backends.cudnn.allow_tf32 = True
      torch.backends.cudnn.benchmark = True
  logging.eval_log_interval=2000    # for 200k runs (logging cadence, not the model)
If "conv rotate" wins by >10%, say the word and osc_core gets the swap behind an
exact-equivalence test, so it is the same model to the bit.
